# NorESM Particle Number Size Distributions (PNSD)

**Scientific question:** What do the modelled aerosol size distributions look like at each station, and how do they relate to the CCN–CDNC **susceptibility** as a function of particle diameter? Overlaying dN/dlogD on the susceptibility curve shows which part of the size spectrum drives the cloud response.

PNSD is reconstructed from NorESM's lognormal modes (SIGMA/NMR/NCONC per mode) via `Function.dNdlogD`. Susceptibility is loaded from the pooled-level binned result.

---
*Sections: 1 Setup · 2 Load data · 3 Build PNSD · 4 Susceptibility + PNSD overlay · 5 Seasonal size distributions.*

## 1 · Setup and configuration

In [ ]:
import PeterChurchillFunctions as Function
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Configuration
NOR_PATH  = "/share/sabl0586/all_stations_NorESM_OsloAero_prcp2szdst_f19_f19_noresmv211_corr_ilevall_levs_4Peter.nc"
SUSC_PATH = "/share/pech2273/NorESM_Susceptibility_All_level.nc"   # pooled (station, radius) slopes

RADII  = np.logspace(0, 2.7, 31)          # cutoff radii [nm], matches the susceptibility file
XSPACE = xr.DataArray(np.logspace(-0.5, 3, num=50), dims=['R'],
                      coords={'R': np.logspace(-0.5, 3, num=50)})   # diameter grid for dNdlogD
N_MODES = 16                              # SIGMA/NMR/NCONC mode indices to scan

## 2 · Load NorESM data and susceptibility

`ds` holds the lognormal mode variables (SIGMA, NMR, NCONC) and Z3 (height). `reg_ds` is the pooled-level binned susceptibility (slope / r_value / std_err per station, radius).

In [ ]:
ds = xr.open_dataset(NOR_PATH, chunks={})
radii   = RADII
height  = ds['Z3'].mean('time')          # (station, lev) mean geopotential height [m]
reg_ds  = xr.open_dataset(SUSC_PATH)      # (station, radius): slope, r_value, std_err

## 3 · Build the particle number size distribution

For each lognormal mode, dN/dlogD = `Function.dNdlogD(NCONC, diameter, NMR, SIGMA)`. Summing the modes gives the total size distribution. NMR is the number-median radius, so the function uses `mu*2` internally for diameter.

In [ ]:
PNSD_ds = xr.Dataset()
for i in range(N_MODES):
    sig, nmr, nconc = f'SIGMA{i:02d}', f'NMR{i:02d}', f'NCONC{i:02d}'
    if sig not in ds:
        continue
    PNSD_ds[sig], PNSD_ds[nmr], PNSD_ds[nconc] = ds[sig], ds[nmr], ds[nconc]
    PNSD_ds[f'dNdlogD{i:02d}'] = xr.apply_ufunc(
        Function.dNdlogD,
        ds[nconc], XSPACE, ds[nmr], ds[sig],
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float],
    )

# Sum modes -> total dN/dlogD, then materialize
dNdlogD_vars = [v for v in PNSD_ds.data_vars if v.startswith('dNdlogD')]
dNdlogD = sum(PNSD_ds[v] for v in dNdlogD_vars).compute()

## 4 · Susceptibility + size distribution overlay

Twin-axis plot: susceptibility (with 95% CI) and correlation on the left, the near-surface size distribution (bottom 6 levels, IQR shaded) on the right — both vs particle diameter.

### 4a · Single station (SMR-II)

In [ ]:
st = 'SMR-II'
fig, ax1 = plt.subplots()

ax1.plot(radii * 2, reg_ds['slope'].sel(station=st), label='Susceptibility')
ax1.fill_between(radii * 2,
    reg_ds['slope'].sel(station=st) - 1.98 * reg_ds['std_err'].sel(station=st),
    reg_ds['slope'].sel(station=st) + 1.98 * reg_ds['std_err'].sel(station=st),
    alpha=0.5, label='95% CI')
ax1.plot(radii * 2, reg_ds['r_value'].sel(station=st), '--', label='Correlation (r)')
ax1.set_xlabel('Particle Diameter $D_p$ (nm)'); ax1.set_ylabel('Susceptibility / Correlation')
ax1.set_xscale('log'); ax1.set_xlim([1, 1000]); ax1.set_ylim([-0.2, 1])

# near-surface PNSD (bottom 6 levels)
pnsd_st = dNdlogD.sel(station=st).isel(lev=slice(-1, -7, -1)).mean('lev')
ax2 = ax1.twinx()
ax2.plot(dNdlogD['R'] * 2, pnsd_st.mean('time'), color='black', label='dN/dlogD')
ax2.fill_between(dNdlogD['R'] * 2,
    pnsd_st.quantile(.25, dim='time'), pnsd_st.quantile(.75, dim='time'),
    color='black', alpha=0.2, label='dN/dlogD IQR')
ax2.set_ylabel('dN / dlogD'); ax2.set_yscale('log'); ax2.set_ylim([.1, 5000])

l1, lab1 = ax1.get_legend_handles_labels(); l2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lab1 + lab2, bbox_to_anchor=(0.5, 0.4))
plt.title(f'CCN-CDNC Susceptibility ({st})'); plt.show()

### 4b · All stations grid

In [ ]:
stations = reg_ds.station.values
fig, axes = plt.subplots(4, 4, figsize=(14, 14), sharex=True, sharey=True)
axes = axes.flatten()

for i, station in enumerate(stations):
    ax = axes[i]
    slope = reg_ds['slope'].sel(station=station)
    std_err = reg_ds['std_err'].sel(station=station)
    ax.plot(radii * 2, slope, label='Susceptibility')
    ax.fill_between(radii * 2, slope - 1.96 * std_err, slope + 1.96 * std_err,
                    alpha=0.5, label='95% CI')
    ax.plot(radii * 2, reg_ds['r_value'].sel(station=station), '--', label='Correlation (r)')
    ax.set_title(station, fontsize=10)
    ax.set_xscale('log'); ax.set_xlim([1, 1000]); ax.set_ylim([-0.2, 1])

    pnsd_st = dNdlogD.sel(station=station).isel(lev=slice(-1, -7, -1)).mean('lev')
    ax2 = ax.twinx()
    ax2.plot(dNdlogD['R'] * 2, pnsd_st.mean('time'), color='black', alpha=0.6, label='PNSD')
    ax2.fill_between(dNdlogD['R'] * 2,
        pnsd_st.quantile(.25, dim='time'), pnsd_st.quantile(.75, dim='time'),
        color='black', alpha=0.2)
    ax2.set_yscale('log'); ax2.set_ylim([10, 5000]); ax2.tick_params(axis='y', labelsize=8)

for ax in axes[12:16]:
    ax.set_xlabel('Particle Diameter $D_p$ (nm)')
for ax in axes[::4]:
    ax.set_ylabel('Susceptibility / Correlation')
fig.text(0.92, 0.5, 'dN / dlogD', va='center', rotation=-90)
plt.tight_layout(rect=[0, 0, 0.9, 1]); plt.show()

## 5 · Seasonal aerosol size distributions (height vs diameter)

Vertical–spectral view at SGP: dN/dlogD as a function of diameter and height, split by season. Shows how the size distribution changes with altitude.

In [ ]:
STATION = 'SGP'

def season_pnsd(months):
    return dNdlogD.sel(station=STATION,
                       time=dNdlogD['time'].dt.month.isin(months)).mean('time')

spring = season_pnsd([3, 4, 5])
summer = season_pnsd([6, 7, 8])

In [ ]:
for label, field in [('Spring', spring), ('Summer', summer)]:
    plt.figure(figsize=(7, 5))
    mesh = plt.pcolormesh(PNSD_ds['R'] * 2, height.sel(station=STATION), field,
                          shading='nearest', cmap='viridis', vmin=0, vmax=5000)
    plt.xscale('log'); plt.xlim([3, 400]); plt.ylim([200, 2500])
    plt.xticks([3, 10, 50, 100, 200], ['3', '10', '50', '100', '200'])
    plt.colorbar(mesh, label='dN/dlogD (cm$^{-3}$)')
    plt.xlabel('Diameter (nm)'); plt.ylabel('Height (m)')
    plt.title(f'{label} Aerosol Size Distribution ({STATION})')
    plt.tight_layout(); plt.show()